In [12]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel 

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
icd_data = pd.read_excel("icd_and_ops/icd_codes_2020_2021.xlsx",
                   usecols=['new name','new code']) 
print('ok')


ok


In [14]:
print(icd_data)

                                                new name new code
0        Bestimmte infektiöse und parasitäre Krankheiten        I
1                             Infektiöse Darmkrankheiten  A00-A09
2                                                Cholera      A00
3      Cholera durch Vibrio cholerae O:1, Biovar chol...    A00.0
4        Cholera durch Vibrio cholerae O:1, Biovar eltor    A00.1
...                                                  ...      ...
11979                                                NaN      NaN
11980                                                NaN      NaN
11981                                                NaN      NaN
11982                                                NaN      NaN
11983                                                NaN      NaN

[11984 rows x 2 columns]


In [15]:
tokenizer = AutoTokenizer.from_pretrained("permediq/SapBERT-DE", use_fast=True)
model = AutoModel.from_pretrained("permediq/SapBERT-DE").to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [16]:
bs = 32 # batch size 
all_embs = []
print(type(icd_data['new name'][0]))
print(type(list(icd_data['new name'][0:32])))
icd_data = icd_data.dropna(subset=['new name'])
print(icd_data['new name'].map(type).value_counts())

for i in tqdm(np.arange(0, len(icd_data), bs)):
    toks = tokenizer(list(icd_data['new name'][i:i+bs]), 
                    padding="max_length", 
                    max_length=40, # model trained with 40 max_length 
                    truncation=True,
                    return_tensors="pt")
    
    tokse = {}
    for k,v in toks.items():
        tokse[k] = v.to(device)
    cls_rep = model(**tokse)[0][:,0,:] 
    all_embs.append(cls_rep.cpu().detach())
all_embs = torch.cat(all_embs)


<class 'str'>
<class 'list'>
new name
<class 'str'>    11776
Name: count, dtype: int64


100%|██████████| 368/368 [02:50<00:00,  2.16it/s]


In [17]:
def cos_sim(a, b):
    a_norm = torch.nn.functional.normalize(a, p=2, dim=1)
    b_norm = torch.nn.functional.normalize(b, p=2, dim=1)
    return torch.mm(a_norm, b_norm.transpose(0, 1))

# cosine similarity of first entity with all the entities



In [18]:
testing = cos_sim(all_embs[0].unsqueeze(0), all_embs)
print(testing)
print(np.argmax(testing))

tensor([[1.0000, 0.5159, 0.3605,  ..., 0.2652, 0.3084, 0.2646]])
tensor(0)


In [20]:
x='heart'
this_embed =[]
toks = tokenizer(x,
                 padding = 'max_length',
                 max_length =40,
                 truncation=True,
                 return_tensors='pt')
tokse = {}
for k,v in toks.items():
    tokse[k] = v.to(device)
cls_rep = model(**tokse)[0][:,0,:] 
this_embed.append(cls_rep.cpu().detach())
this_embed = torch.cat(this_embed)
closest = cos_sim(this_embed, all_embs)
print(closest)
k = np.argmax(closest).item()
print(k)
print(icd_data['new name'][k])
print(icd_data['new code'][k])

tensor([[0.2339, 0.2143, 0.2001,  ..., 0.2537, 0.2719, 0.2421]])
1078
Extrahepatischer Gallengang
C24.0
